# 02 - Creating the Machine Learning Dataset

**Goal of this notebook:** turn the raw visibility cube into a folder of
small, labeled image patches that a CNN can be trained on.

You will:
1. Load a configuration from `configs/dataset.yaml`.
2. Extract amplitude/label patches for every baseline.
3. Filter patches (e.g. discard patches with no RFI at all).
4. Normalize patches (min-max / z-score / log).
5. Split into train / validation / test sets.
6. Save everything to disk, along with a `metadata.csv` describing every patch.
7. Visualize a few examples of each split, and inspect the class balance.


## 1. Setup

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data.loader import load_observation, make_synthetic_observation
from data.dataset import (
    DatasetConfig, build_patches, split_patches, export_dataset, create_dataset,
    inspect_rfi_fraction_distribution,
)
from data.visualization import plot_waterfall, plot_mask, plot_overlay

%matplotlib inline


## 2. Load the configuration and the observation

In [ ]:
config = DatasetConfig.from_yaml("../configs/dataset.yaml")
print(config)


In [ ]:
USE_REAL_DATA = False
RDB_PATH = "meerkat_data.rdb"

if USE_REAL_DATA:
    observation = load_observation(RDB_PATH)
else:
    print("Using synthetic data (set USE_REAL_DATA = True to load a real .rdb file).")
    observation = make_synthetic_observation(n_time=512, n_freq=512, n_baseline=4)


## 3. Sanity-check: does the patch shape actually fit this observation?

Before extracting anything, it's worth checking that the configured
patch shape is not bigger than the observation itself. This matters a
lot more with real data than with the synthetic example: a short
calibration "track" scan might only have a few dozen time dumps, well
under a typical `patch_size` of 128. If that happens, **no patch can
ever be extracted from any baseline** - not because there's no RFI, but
because the sliding window used to cut out patches never fits even once.

Real data is also often very *asymmetric*: e.g. `n_time=22` but
`n_freq=4096` is a totally normal shape for a short scan. Forcing a
single square `patch_size` in that case is a bad fit either way - too
big for the time axis, or (if shrunk to fit) wasteful of all that
frequency resolution. `configs/dataset.yaml` supports independent
`patch_size` (time) and `patch_size_freq` (frequency) for exactly this
situation - see the commented-out example in that file.

`build_patches` (and the lower-level `extract_patches`) will raise a
clear `ValueError` if the configured shape doesn't fit, but it's better
to catch it here first with a plain comparison.


In [ ]:
patch_time = config.patch_size
patch_freq = config.patch_size_freq if config.patch_size_freq is not None else config.patch_size

print(f"Observation:  n_time={observation.n_time}, n_freq={observation.n_freq}")
print(f"Config:       patch_size (time)={patch_time}, patch_size_freq={patch_freq}, "
      f"stride={config.stride}, stride_freq={config.stride_freq}")

problem = False
if patch_time > observation.n_time:
    problem = True
    print(f"\nWARNING: patch_size (time)={patch_time} is larger than n_time={observation.n_time}.")
if patch_freq > observation.n_freq:
    problem = True
    print(f"\nWARNING: patch_size_freq={patch_freq} is larger than n_freq={observation.n_freq}.")

if problem:
    print(
        f"\nNo patches could be extracted as-is. Fix in configs/dataset.yaml:\n"
        f"  patch_size:      at most {observation.n_time}   (time axis)\n"
        f"  patch_size_freq: at most {observation.n_freq}  (frequency axis)\n"
        f"If n_time and n_freq are very different (as here), prefer setting "
        f"patch_size and patch_size_freq to different values (a rectangular "
        f"patch) rather than forcing a single square patch_size - see the "
        f"commented example in configs/dataset.yaml. Then reload `config` "
        f"above and re-run this cell."
    )
else:
    print("\nLooks good - the configured patch shape fits within this observation's dimensions.")


## 4. IMPORTANT: check the RFI fraction distribution before filtering

Real MeerKAT observations are often **much more sparsely and unevenly
flagged** than the small synthetic example used to test this pipeline.
If you jump straight to filtering with the default
`min_rfi_fraction` / `discard_empty` settings, it is easy to end up with
**zero patches** and no clear explanation why - the filters may simply
be stricter than the actual RFI density in your data.

`inspect_rfi_fraction_distribution` extracts every patch with **no
filtering applied** and returns the per-patch flagged fraction, so you
can see the real distribution *before* deciding on thresholds.


In [ ]:
raw_fractions = inspect_rfi_fraction_distribution(observation, config)

print(f"Total (unfiltered) patches: {len(raw_fractions)}")
print(f"Patches with 0% RFI:        {(raw_fractions == 0).sum()}")
print(f"Patches with any RFI:       {(raw_fractions > 0).sum()} ({(raw_fractions > 0).mean() * 100:.2f}%)")
print(f"Mean flagged fraction:      {raw_fractions.mean():.5f}")
print(f"Max flagged fraction:       {raw_fractions.max():.5f}")
print(f"95th percentile:            {np.percentile(raw_fractions, 95):.5f}")
print(f"99th percentile:            {np.percentile(raw_fractions, 99):.5f}")

plt.figure(figsize=(6, 4))
plt.hist(raw_fractions, bins=50)
plt.xlabel("Fraction of RFI pixels in patch (unfiltered)")
plt.ylabel("Number of patches")
plt.title("Raw per-patch RFI fraction, before any filtering")
plt.show()


**How to use this histogram:** pick `min_rfi_fraction` (in
`configs/dataset.yaml`) to sit comfortably *below* the bulk of your
non-zero patches - for real data this is often much smaller than the
`0.01` (1%) default, e.g. `0.0001`-`0.001`, or even `0.0` combined with
`discard_empty: true` (which only removes patches with *literally no*
RFI, rather than requiring a minimum density). If the printed "Patches
with any RFI" count is already 0, the flags themselves may not be
reaching this baseline/patch combination at all - double check the
`pol` / `corrprods` / `scans` selection used in `load_observation`, and
compare against what you saw in `01_explore_data.ipynb`.

Once you're happy with the thresholds, re-run the config cell above (or
edit `configs/dataset.yaml` and reload it) before continuing.


## 5. Build the dataset step by step

The convenience function `create_dataset(...)` does everything in one call
(see the bottom of this notebook), but let's first walk through the
individual steps so you can see exactly what's happening at each stage.


In [ ]:
# Step 1: extract, filter, and normalize patches for every baseline
patches = build_patches(observation, config)
print(f"Total patches kept after filtering: {len(patches)}")

flagged_fractions = [p.patch.flagged_fraction for p in patches]
print(f"Mean flagged fraction per patch: {np.mean(flagged_fractions):.4f}")
print(f"Patches with 0% RFI:   {sum(f == 0.0 for f in flagged_fractions)}")
print(f"Patches with 100% RFI: {sum(f == 1.0 for f in flagged_fractions)}")


In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(flagged_fractions, bins=30)
plt.xlabel("Fraction of RFI pixels in patch")
plt.ylabel("Number of patches")
plt.title("Distribution of RFI fraction across patches")
plt.show()


In [ ]:
# Step 2: split into train / validation / test
split_dict = split_patches(patches, config)
for split_name, split_patches_list in split_dict.items():
    print(f"{split_name:12s}: {len(split_patches_list)} patches")


In [ ]:
# Step 3: write everything to disk (images/, labels/, metadata.csv)
metadata_path = export_dataset(split_dict, config, overwrite=True)
print(f"Dataset written. Metadata file: {metadata_path}")


## 6. Inspect the exported dataset

In [ ]:
metadata = pd.read_csv(metadata_path)
metadata.head()


In [ ]:
metadata.groupby("split").agg(
    n_patches=("filename", "count"),
    mean_flagged_fraction=("flagged_fraction", "mean"),
)


### Visualize a few random patches from the training set

In [ ]:
train_rows = metadata[metadata["split"] == "train"].sample(3, random_state=0)

fig, axes = plt.subplots(len(train_rows), 3, figsize=(14, 4 * len(train_rows)))
for row_idx, (_, row) in enumerate(train_rows.iterrows()):
    image = np.load(f"dataset/train/images/img_{row['filename']}")
    label = np.load(f"dataset/train/labels/lbl_{row['filename']}")

    plot_waterfall(image, title=f"Image ({row['filename']})", ax=axes[row_idx, 0])
    plot_mask(label, title="Label mask", ax=axes[row_idx, 1])
    plot_overlay(image, label, title="Overlay", ax=axes[row_idx, 2])

fig.tight_layout()
plt.show()


## 7. One-liner alternative

Once you're happy with the config, the whole pipeline above can be run in
a single call:

```python
metadata_path = create_dataset(observation, config, overwrite=True)
```

This is exactly what `main.py` does from the command line:

```bash
python main.py --synthetic --config configs/dataset.yaml --overwrite
# or, with real data:
python main.py --rdb-path meerkat_data.rdb --config configs/dataset.yaml
```


## 8. Things to try

- Change `normalization` in `configs/dataset.yaml` (minmax / zscore / log) and re-run - how does the visualized image change?
- Change `patch_size` and `stride` - how does the number of patches change? (Hint: use `data.patching.count_patches` to predict it first.)
- Try `discard_empty: true` vs `false` - how does the class balance in the histogram change?
- If you're working with real data and hit a `ValueError` about `patch_size` being larger than the observation - see section 3 - lower `patch_size` to fit your data's actual number of time dumps/frequency channels.
- If patches ARE extracted but you still get 0 kept after filtering, re-check the histogram in section 4 - `min_rfi_fraction` is very likely set well above the actual RFI density in your observation. Try lowering it (e.g. `0.001` or `0.0`) and re-running.
- Once you're happy with the dataset, move on to **`03_train_cnn.ipynb`** (Part 2 of the workshop) to train a CNN on it.
